# 评分卡高级功能演示

本 Notebook 演示 scorecardpipeline 项目三个高级功能模块：

1. **SHAP 模型解释性** — 基于 SHAP 值对评分卡模型进行特征归因分析，支持单样本解释和全局特征重要度汇总。
2. **模型监控** — 上线后稳定性监控，包括评分分布 PSI、特征 PSI 以及模型性能（KS/AUC）衰减检测。
3. **概率校准** — 使用 Platt Scaling 和 Isotonic Regression 校准模型预测概率，使其更贴近真实违约率。

数据集使用内置的 `germancredit`（德国信贷数据集），标签 `creditability` 映射为 0（good）/ 1（bad）。

## 环境准备

导入本项目所有需要用到的模块。`scorecardpipeline` 包已将 `ScorecardExplainer`、`ModelMonitor`、`ProbabilityCalibrator` 暴露在顶层命名空间，可直接导入使用。

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from scorecardpipeline import (
    FeatureSelection,
    Combiner,
    WOETransformer,
    ITLubberLogisticRegression,
    germancredit,
    ScorecardExplainer,
    ModelMonitor,
    ProbabilityCalibrator,
)

# 设置中文字体显示
plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

## 数据准备

加载 `germancredit` 数据集，将标签 `creditability` 从文本映射为 0/1（good → 0, bad → 1），并按 7:3 划分训练集与测试集，采用分层抽样保持正负样本比例一致。

In [ ]:
target = "creditability"

# 加载德国信贷数据集
data = germancredit()

# 标签映射：good -> 0, bad -> 1
data[target] = data[target].map({"good": 0, "bad": 1})

# 分层划分训练集与测试集
train, test = train_test_split(data, test_size=0.3, random_state=42, stratify=data[target])

print(f"训练集样本量: {len(train)}，坏样本率: {train[target].mean():.4f}")
print(f"测试集样本量: {len(test)}，坏样本率: {test[target].mean():.4f}")
train.head()

## 构建 Pipeline

构建经典评分卡数据处理流水线：

1. **FeatureSelection** — 基于 scorecardpy 引擎进行特征筛选（IV 值过滤、空值率过滤、相关性过滤等）。
2. **Combiner** — 使用卡方分箱方法（`method="chi"`），最大分箱数 4，单箱最小样本占比 5%。
3. **WOETransformer** — 将分箱结果转换为 WOE 编码值。

Pipeline 先在训练集上 `fit`，再对训练集和测试集分别 `transform`，得到 WOE 编码后的特征矩阵。

In [ ]:
# 构建评分卡数据处理流水线
pipeline = Pipeline([
    ("select", FeatureSelection(target=target, engine="scorecardpy", iv=0.02)),
    ("combiner", Combiner(target=target, method="chi", min_bin_size=0.05, max_n_bins=4)),
    ("woe", WOETransformer(target=target)),
])

# 在训练集上拟合流水线
pipeline.fit(train)

# 转换训练集与测试集为 WOE 编码
woe_train = pipeline.transform(train)
woe_test = pipeline.transform(test)

print(f"WOE 编码后训练集特征数: {woe_train.shape[1] - 1}（不含标签列）")
print(f"入选特征列表: {list(woe_train.drop(columns=[target]).columns)}")
woe_train.head()

## 训练 ITLubberLogisticRegression

使用 `ITLubberLogisticRegression` 训练逻辑回归模型。该类继承自 `sklearn.linear_model.LogisticRegression`，额外提供系数标准误、P 值、VIF 等统计信息输出能力。

模型的 `fit` 方法会自动从数据中提取目标列，训练完成后可通过 `predict_proba` 进行预测。注意调用 `predict_proba` 时需去掉目标列。

In [ ]:
# 训练逻辑回归模型
model = ITLubberLogisticRegression(target=target)
model.fit(woe_train)

# 预测训练集与测试集的正类概率
score_train = model.predict_proba(woe_train.drop(columns=[target]))[:, 1]
score_test = model.predict_proba(woe_test.drop(columns=[target]))[:, 1]

print(f"训练集预测概率范围: [{score_train.min():.4f}, {score_train.max():.4f}]")
print(f"测试集预测概率范围: [{score_test.min():.4f}, {score_test.max():.4f}]")

# 输出模型统计摘要（系数、标准误、P 值、VIF 等）
model.summary()

## SHAP 模型解释性

使用 `ScorecardExplainer` 对训练好的评分卡模型进行 SHAP 解释。该解释器接收原始特征 DataFrame，内部自动完成分箱 → WOE 转换 → SHAP 归因的完整链路。

支持三种解释输出：
- `explain(X)` — 批量计算所有样本的 SHAP 值矩阵
- `explain_sample(X, index)` — 单样本解释，输出每个特征的 SHAP 贡献与原始取值
- `summary(X)` — 全局特征重要度汇总（按 SHAP 绝对值均值排序）

In [ ]:
# 初始化评分卡解释器
explainer = ScorecardExplainer(
    model=model,
    combiner=pipeline.named_steps["combiner"],
    woe_transformer=pipeline.named_steps["woe"],
)

# 单样本解释：输出每个原始特征对该样本预测结果的 SHAP 贡献
sample_explanation = explainer.explain_sample(test, index=0)
print("=== 单样本 SHAP 解释（index=0）===")
sample_explanation

In [ ]:
# 全局特征重要度汇总：按 SHAP 绝对值均值降序排列
importance = explainer.summary(test)
print("=== 特征重要度汇总（mean_abs_shap 降序）===")
importance

# 可视化特征重要度
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance["feature"], importance["mean_abs_shap"], color="steelblue")
ax.set_xlabel("mean(|SHAP value|)")
ax.set_title("特征重要度 (SHAP)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 模型监控

使用 `ModelMonitor` 对模型上线后的稳定性进行监控。该模块提供三类监控能力：

1. **评分分布 PSI** — `score_psi(score)` 计算当前评分分布与基准分布之间的 PSI（Population Stability Index）。
2. **特征 PSI** — `feature_psi(X)` 计算各特征分布的漂移程度，识别哪些特征发生了显著变化。
3. **性能衰减** — `performance_decay(score, y_true)` 对比当前数据与基准数据的 KS/AUC 差异。

使用流程：先在训练集上 `fit_reference` 建立监控基准，再在测试集上计算各项监控指标。

In [ ]:
# 初始化模型监控器，评分分 10 箱
monitor = ModelMonitor(score_bins=10)

# 在训练集上建立监控基准
monitor.fit_reference(train, score_train, y_true=train[target].values)

# 1. 评分分布 PSI
psi_score = monitor.score_psi(score_test)
print(f"评分分布 PSI: {psi_score:.6f}")

# 2. 特征 PSI
feature_psi = monitor.feature_psi(test)
print("\n=== 各特征 PSI ===")
feature_psi

In [ ]:
# 3. 模型性能衰减监控
metrics = monitor.performance_decay(score_test, test[target].values)

print("=== 模型性能衰减 ===")
for key, value in metrics.items():
    print(f"  {key}: {value:.6f}")

# 将性能指标整理为 DataFrame 方便后续保存
metrics_df = pd.DataFrame([metrics])
metrics_df

## 概率校准

评分卡模型训练完成后，预测概率可能存在系统性偏差（如整体偏高或偏低）。`ProbabilityCalibrator` 提供两种经典校准方法：

- **Platt Scaling** — 用一个逻辑回归模型对原始概率进行再拟合，适合 sigmoid 形状的偏差校正。
- **Isotonic Regression** — 非参数保序回归，更灵活，适合非线性单调映射，但需要较多样本。

下方分别用两种方法在训练集上拟合校准器，并对测试集概率进行校准，随后绘制校准前后概率分布对比图。

In [ ]:
# Platt Scaling 校准
calibrator_platt = ProbabilityCalibrator(method="platt")
calibrated_train_platt = calibrator_platt.fit_transform(score_train, train[target].values)
calibrated_test_platt = calibrator_platt.transform(score_test)

# Isotonic Regression 校准
calibrator_isotonic = ProbabilityCalibrator(method="isotonic")
calibrated_train_iso = calibrator_isotonic.fit_transform(score_train, train[target].values)
calibrated_test_iso = calibrator_isotonic.transform(score_test)

print("=== 校准前后概率统计（测试集）===")
calibration_summary = pd.DataFrame({
    "原始概率": score_test,
    "Platt 校准": calibrated_test_platt,
    "Isotonic 校准": calibrated_test_iso,
}).describe()
calibration_summary

# 绘制校准前后概率分布对比图
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(score_test, bins=30, color="steelblue", alpha=0.7, edgecolor="white")
axes[0].set_title("原始概率分布")
axes[0].set_xlabel("预测概率")
axes[0].set_ylabel("样本数")

axes[1].hist(calibrated_test_platt, bins=30, color="seagreen", alpha=0.7, edgecolor="white")
axes[1].set_title("Platt Scaling 校准后")
axes[1].set_xlabel("预测概率")

axes[2].hist(calibrated_test_iso, bins=30, color="coral", alpha=0.7, edgecolor="white")
axes[2].set_title("Isotonic Regression 校准后")
axes[2].set_xlabel("预测概率")

plt.suptitle("概率校准前后分布对比（测试集）", fontsize=14)
plt.tight_layout()
plt.show()

## 结果输出

将以上各步骤产出的结果表格统一保存到 `examples/model_report/` 目录，便于后续查阅与汇报。

In [ ]:
# 创建输出目录
output_dir = "examples/model_report"
os.makedirs(output_dir, exist_ok=True)

# 保存 SHAP 解释结果
sample_explanation.to_excel(f"{output_dir}/shap_sample_explanation.xlsx", index=False)
importance.to_excel(f"{output_dir}/shap_feature_importance.xlsx", index=False)

# 保存模型监控结果
feature_psi.to_excel(f"{output_dir}/model_monitor_feature_psi.xlsx", index=False)
metrics_df.to_excel(f"{output_dir}/model_monitor_performance_decay.xlsx", index=False)

# 保存概率校准结果
calibration_summary.to_excel(f"{output_dir}/probability_calibration_summary.xlsx")

print(f"所有结果已保存至 {output_dir}/ 目录")
for f in sorted(os.listdir(output_dir)):
    if f.endswith(".xlsx"):
        print(f"  - {f}")